# Module 01: The Prompt Engineering Layer — Steering the Agentic Brain

This notebook explores the foundations of directing Large Language Models (LLMs) to act as the core reasoning engine for autonomous agents. Prompt engineering is an iterative process: inadequate prompts can lead to ambiguous or inaccurate responses. This curriculum emphasizes building from scratch, using plain Python and Pydantic for reliability and transparency.

## 1. Environment Setup

Since we are building for production-grade reliability, we will use Pydantic for state validation and the Gemini API for reasoning.

In [ ]:
# Install dependencies (uncomment if running in Colab or a fresh environment)
# !pip install -q -U google-generativeai pydantic

import os
import google.generativeai as genai
from pydantic import BaseModel, Field
from typing import List, Optional

# Setup API Key (Replace with your actual key or use Colab Secrets)
# os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

## 2. System & Role Prompting

Crafting effective prompts involves different layers of instruction.

- **System Prompting:** Establishes the "big picture," defining the model’s overall purpose and constraints.
- **Role Prompting:** Assigns a specific identity or persona to ensure consistent domain expertise.

In [ ]:
class AgentConfig(BaseModel):
    persona: str = "You are a Senior Software Engineer specializing in Python and Pydantic."
    constraints: List[str] = [
        "Always return valid JSON.",
        "Do not use external libraries like LangChain.",
        "Be concise and technical."
    ]

def get_system_instruction(config: AgentConfig):
    return f"{config.persona}\nConstraints: {', '.join(config.constraints)}"

# Example Usage
config = AgentConfig()
print(get_system_instruction(config))

### Contextual Prompting

Contextual prompts supply specific background details relevant to a task, helping the model handle nuances. In an agentic system, you are the director providing the "scene" and data.

## 3. Output Configurations

Managing the non-deterministic nature of LLMs is critical for reliability.
- **Temperature:** Controls randomness ($0.0$ is deterministic, higher values are more "creative").
- **Top-K:** The model chooses from the top $K$ most likely tokens.
- **Top-P:** Nucleus sampling; the model chooses from a set of tokens whose cumulative probability exceeds $P$.

In [ ]:
# Configuration for a deterministic "Brain"
generation_config = {
    "temperature": 0.0,
    "top_p": 0.95,
    "top_k": 40,
    "max_output_tokens": 1024,
    "response_mime_type": "application/json",
}

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    generation_config=generation_config
)

## 4. Advanced Reasoning Patterns

To move beyond simple chat, we implement loops that allow the model to "think".

- **Chain-of-Thought (CoT):** Elicits step-by-step reasoning before providing a final answer.
- **ReAct (Reason & Act):** Synergizes reasoning and acting. The agent "Thinks" about a problem, "Acts" by calling a tool, and "Observes" the result before making the next move.

In [ ]:
class ReActStep(BaseModel):
    thought: str = Field(description="The internal reasoning of the agent.")
    action: str = Field(description="The tool to call.")
    action_input: str = Field(description="The parameters for the tool.")

# This Pydantic model ensures the 'Brain' returns a structured trajectory

## 5. Hands-on Challenge: The "First Principles" Orchestrator

**Challenge:** Create a prompt that forces the model to use the ReAct pattern to solve a math word problem using only a hypothetical calculator tool.

**Rules:**
- Use a System Prompt to define the persona.
- Use CoT to ensure it explains its math.
- Validate the output using the `ReActStep` Pydantic model.

In [ ]:
# Example: Prompt for a math word problem using ReAct
system_prompt = get_system_instruction(config)
cot_instruction = "Think step by step and explain your reasoning before acting."
react_instruction = (
    "Use the ReAct pattern: For each step, provide your 'thought', specify the 'action' (use only the 'calculator' tool), "
    "and the 'action_input' (the calculation to perform). Return your output as a valid JSON object conforming to the ReActStep model."
)

math_problem = "If a train travels 60 miles per hour for 2.5 hours, how far does it go?"

full_prompt = f"""{system_prompt}\n\nTask: {math_problem}\n\nInstructions:\n- {cot_instruction}\n- {react_instruction}\n"""

print(full_prompt)
# (In practice, you would send 'full_prompt' to the model and validate the output with ReActStep)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/01_Prompt_Engineering_Layer/prompt_engineering_layer.ipynb)

*This notebook encourages a first-principles, curriculum-first approach to agentic engineering. Build, test, and iterate using plain Python and Pydantic for maximum transparency and reliability.*